# 模板渲染
Starlette没有固定指定一种模板引擎来渲染输出HTML内容，但是在日常使用中，Jinja2模板引擎是比较好的一个匹配选择。Starlette针对Jinja2提供了简便的配置方法。以下是一个使用模板渲染输出的示例。

In [ ]:
from starlette.applications import Starlette
from starlette.staticfiles import StaticFiles
from starlette.templating import Jinja2Templates


templates = Jinja2Templates(directory="templates")

app = Starlette()
app.mount("/static", StaticFiles(directory="statics"), name="static")

@app.route("/")
async def homepage(request):
	return templates.TemplateResponse("index.html", { "request": request })

首先需要对`Jinja2Templates`类进行实例化，获取模板引擎。实例化时需要指定模板文件所在目录。`Jinja2Templates`类的实例可以用来渲染模板目录中的指定模板。渲染模板需要使用`TemplateResponse()`方法，这个方法可以接受以下参数，并直接产生响应输出。

- `name`，模板文件名。
- `context`，要传递给模板的上下文实例，一般采用字典类型。
- `status_code`，输出响应的Status Code。
- `headers`，设置输出响应的头信息。
- `media_type`，设置输出的媒体类型信息。
- `background`，完成响应输出后要执行的后台任务。

Starlette 提供了一种简单的配置方式`jinja2`。这可能是您默认使用的。

In [ ]:
from starlette.applications import Starlette
from starlette.routing import Route, Mount
from starlette.templating import Jinja2Templates
from starlette.staticfiles import StaticFiles


templates = Jinja2Templates(directory='templates')

async def homepage(request):
    return templates.TemplateResponse(request, 'index.html')

routes = [
    Route('/', endpoint=homepage),
    Mount('/static', StaticFiles(directory='static'), name='static')
]

app = Starlette(debug=True, routes=routes)

请注意，传入的request实例必须作为模板上下文的一部分包含在内。

`Jinja2` 模板上下文将自动包含一个`url_for`函数，因此我们可以正确地超链接到应用程序内的其他页面。

例如，我们可以从 HTML 模板中链接到静态文件：

In [1]:
<link href="{{ url_for('static', path='/css/bootstrap.min.css') }}" rel="stylesheet" />

SyntaxError: invalid syntax (3985409910.py, line 1)

如果您想使用自定义过滤器，则需要更新env 以下属性`Jinja2Templates`：

In [ ]:
from commonmark import commonmark
from starlette.templating import Jinja2Templates

def marked_filter(text):
    return commonmark(text)

templates = Jinja2Templates(directory='templates')
templates.env.filters['marked'] = marked_filter

## 使用自定义 jinja2.Environment 实例
Starlette 也接受预配置`jinja2.Environment`实例。

In [ ]:
import jinja2
from starlette.templating import Jinja2Templates

env = jinja2.Environment(...)
templates = Jinja2Templates(env=env)

## 上下文处理器
上下文处理器是一个函数，它返回一个要合并到模板上下文中的字典。每个函数只接受一个参数`request`，并且必须返回一个要添加到上下文中的字典。

模板处理器的一个常见用例是使用共享变量扩展模板上下文。

In [ ]:
import typing
from starlette.requests import Request

def app_context(request: Request) -> typing.Dict[str, typing.Any]:
    return {'app': request.app}

### 注册上下文模板
将上下文处理器传递给 Jinja2Templates 类的 `context_processors` 参数。

In [ ]:
import typing

from starlette.requests import Request
from starlette.templating import Jinja2Templates

def app_context(request: Request) -> typing.Dict[str, typing.Any]:
    return {'app': request.app}

templates = Jinja2Templates(
    directory='templates', context_processors=[app_context]
)

## 测试模板响应
当使用测试客户端时，模板响应包括`.template`和`.context` 属性。

In [ ]:
from starlette.testclient import TestClient


def test_homepage():
    client = TestClient(app)
    response = client.get("/")
    assert response.status_code == 200
    assert response.template.name == 'index.html'
    assert "request" in response.context

## 自定义 Jinja2 环境

`Jinja2Templates` 接受 `Jinja2 Environment` 支持的所有选项。这将允许对 `Starlette` 创建的环境实例进行更多控制。

有关环境可用选项的列表，请点击此处查看 [Jinja2 文档](https://jinja.palletsprojects.com/en/3.0.x/api/#jinja2.Environment)

In [ ]:
from starlette.templating import Jinja2Templates


templates = Jinja2Templates(directory='templates', 
                            autoescape=False, 
                            auto_reload=True)

## 异步模板渲染
Jinja2 支持异步模板渲染，但是作为一般规则，我们建议您不要在模板中添加调用数据库查找或其他 I/O 操作的逻辑。

相反，我们建议您确保您的端点执行所有 I/O，例如，严格评估视图中的任何数据库查询并将最终结果包含在上下文中。